<a href="https://colab.research.google.com/github/kamranshahid56-tech/multimodal-disease-prediction-from-chest-x-rays-radiology-reports/blob/main/multimodal_disease_prediction_from_chest_x_rays_%26_radiology_reports.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# MULTIMODAL DISEASE PREDICTION FROM CHEST X-RAYS & RADIOLOGY REPORTS
# Late-Fusion Deep Learning Architecture (DenseNet-121 + ClinicalBERT)
# Dataset: MIMIC-CXR v2.0.0
# Target: 5 Classes — Cardiomegaly, Pleural Effusion, Edema, Pneumothorax, Pneumonia
# ============================================================


In [1]:
# Install all packages not available by default in Colab.
# - transformers: Hugging Face library for ClinicalBERT
# - torchvision: DenseNet-121 and image transforms
# - scikit-learn: metrics (AUROC, F1, confusion matrix)
# - opencv-python: CLAHE image enhancement
# - pandas / numpy: data manipulation
# - matplotlib / seaborn: plotting

!pip install -q transformers torchvision scikit-learn opencv-python-headless pandas numpy matplotlib seaborn tqdm

#CELL 2 — Download Datasets & Set Paths   

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   CELL 2 — Download Both Datasets Directly into Colab       ║
# ╚══════════════════════════════════════════════════════════════╝

import os



BASE_JPG = "https://physionet.org/files/mimic-cxr-jpg/2.0.0"
BASE_CXR = "https://physionet.org/files/mimic-cxr/2.0.0"

os.makedirs('/content/mimic-cxr/files',   exist_ok=True)
os.makedirs('/content/mimic-cxr/reports', exist_ok=True)

# ── FROM MIMIC-CXR-JPG: Labels + Split + Image list (tiny files) ──
print("Step 1/3: Downloading labels and split files...")
for f in ["mimic-cxr-2.0.0-chexpert.csv.gz",
          "mimic-cxr-2.0.0-split.csv.gz",
          "IMAGE_FILENAMES"]:
    !wget -q -N -c --user={PHYSIONET_USER} --password={PHYSIONET_PASS} \
        {BASE_JPG}/{f} -P /content/mimic-cxr/
print("✔ Labels and split downloaded")

# ── FROM MIMIC-CXR: Reports (text for ClinicalBERT) ──────────────
print("Step 2/3: Downloading radiology reports (~1 GB)...")
!wget -q -N -c --user={PHYSIONET_USER} --password={PHYSIONET_PASS} \
    {BASE_CXR}/mimic-cxr-reports.zip \
    -P /content/mimic-cxr/
!unzip -q /content/mimic-cxr/mimic-cxr-reports.zip \
    -d /content/mimic-cxr/reports/
print("✔ Reports downloaded and extracted")

# ── FROM MIMIC-CXR-JPG: Images (subset only to save space) ───────
# Downloads first 10,000 images (~1.5 GB) using IMAGE_FILENAMES trick
# This is enough to train and test the full pipeline
print("Step 3/3: Downloading image subset (~1.5 GB)...")
!head -n 5000 /content/mimic-cxr/IMAGE_FILENAMES | \
    wget -r -N -c -np -nH --cut-dirs=1 \
    --user={PHYSIONET_USER} --password={PHYSIONET_PASS} \
    -i - --base={BASE_JPG}/ \
    -P /content/mimic-cxr/
print("✔ Images downloaded")

# ── Set all paths used by remaining cells ─────────────────────────
IMAGE_DIR  = '/content/mimic-cxr/files'
LABEL_CSV  = '/content/mimic-cxr/mimic-cxr-2.0.0-chexpert.csv.gz'
SPLIT_CSV  = '/content/mimic-cxr/mimic-cxr-2.0.0-split.csv.gz'
REPORT_DIR = '/content/mimic-cxr/reports/files'   # reports extract to files/

print("\n✔ All paths configured. Ready to run remaining cells.")

#CELL 3 — Imports & Global Config

In [3]:
import re
import random
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
# ── Reproducibility seed ──────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {DEVICE}")

In [ ]:
CLASSES = ['Cardiomegaly', 'Pleural Effusion', 'Edema', 'Pneumothorax', 'Pneumonia']
NUM_CLASSES = len(CLASSES)

In [ ]:
# ── Training hyperparameters (from the paper) ─────────────────
BATCH_SIZE   = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS   = 25
MAX_TEXT_LEN = 256    # ClinicalBERT token limit used in the paper
DROPOUT_RATE = 0.3
FC_HIDDEN    = 512    # hidden units in the fusion head
IMAGE_SIZE   = 224    # DenseNet-121 input resolution

#CELL 4 — Data Loading & Label Parsing

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CELL 4 — Data Loading & Label Parsing               ║
# ╚══════════════════════════════════════════════════════════════╝

# BUG 1: Files are .csv.gz not .csv — pandas handles this automatically
#         but the merge key was wrong
# BUG 2: chexpert.csv has NO dicom_id column — only subject_id + study_id
#         so we cannot merge on dicom_id
# BUG 3: Labels include -1.0 (uncertain) which must be handled,
#         not just NaN. clip(0,1) turns -1 into 0 which is WRONG.

labels_df = pd.read_csv(LABEL_CSV)   # pandas reads .gz automatically
split_df  = pd.read_csv(SPLIT_CSV)

# chexpert.csv has: subject_id, study_id, + 14 label columns
# split.csv has:    subject_id, study_id, dicom_id, split
# Merge only on the common keys — NOT dicom_id
df = pd.merge(labels_df, split_df, on=['subject_id', 'study_id'])

# Handle all 3 label values correctly:
#   1.0  → positive (keep as 1)
#  -1.0  → uncertain → treat as 0 (conservative, as done in the paper)
#   0.0  → negative (keep as 0)
#   NaN  → not mentioned → treat as 0
df[CLASSES] = df[CLASSES].replace(-1.0, 0.0).fillna(0.0)

# Remove rows where ALL five target labels are 0
df = df[df[CLASSES].sum(axis=1) > 0].reset_index(drop=True)

# Sanity check
print(f"Total usable rows : {len(df)}")
print(df['split'].value_counts())
print("\nLabel prevalence (% positive):")
print((df[CLASSES].mean() * 100).round(2))

#CELL 5 — Report Text Preprocessing

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CELL 5 — Report Text Preprocessing  (FIXED)        ║
# ╚══════════════════════════════════════════════════════════════╝

# BUG: The actual MIMIC-CXR report path structure is:
#   reports/files/p{prefix}/p{subject_id}/s{study_id}.txt
#                  ↑ prefix is FULL "p10", "p11" etc — NOT just 2 digits
# Original code used subject_str[:2] which gives "10" not "p10"

def load_report(subject_id: int, study_id: int) -> str:
    """
    Load a radiology report text file from MIMIC-CXR.
    Actual path structure after extracting mimic-cxr-reports.zip:
      REPORT_DIR/files/p{p_prefix}/p{subject_id}/s{study_id}.txt
    Where p_prefix = 'p' + first two digits of subject_id
    e.g. subject 10000032 → prefix 'p10' → path: files/p10/p10000032/s{study_id}.txt
    """
    subject_str = str(subject_id)
    p_prefix    = 'p' + subject_str[:2]          # correct: 'p10', 'p11', etc.
    report_path = os.path.join(
        REPORT_DIR, 'files', p_prefix, f'p{subject_id}', f's{study_id}.txt'
    )
    if not os.path.exists(report_path):
        return ""
    with open(report_path, 'r', encoding='utf-8', errors='replace') as f:
        return f.read()


def extract_findings_impression(raw_text: str) -> str:
    """Extract only FINDINGS and IMPRESSION sections from report."""
    text = re.sub(r'\[\*\*.*?\*\*\]', '', raw_text)
    text = re.sub(r'\s+', ' ', text).strip()

    section_pattern = re.compile(
        r'(FINDINGS|IMPRESSION)\s*:?\s*(.*?)(?=(?:FINDINGS|IMPRESSION|COMPARISON|INDICATION|TECHNIQUE|NOTIFICATION)\s*:|$)',
        re.IGNORECASE | re.DOTALL
    )
    matches = section_pattern.findall(text)
    extracted = ' '.join(m[1].strip() for m in matches if m[1].strip())
    return extracted if extracted else text


# Pre-cache all reports
print("Pre-loading reports...")
report_cache = {}
for _, row in tqdm(df.iterrows(), total=len(df)):
    key = (row['subject_id'], row['study_id'])
    if key not in report_cache:
        raw = load_report(row['subject_id'], row['study_id'])
        report_cache[key] = extract_findings_impression(raw)

# Warn if many reports are missing
missing = sum(1 for v in report_cache.values() if v == "")
print(f"Reports cached: {len(report_cache)} | Missing/empty: {missing}")

#CELL 6 — Image Preprocessing & Augmentation

In [ ]:
# The paper applies:
#   1. Resize to 224×224
#   2. Convert to 3-channel (grayscale CXR replicated across RGB)
#   3. CLAHE (Contrast Limited Adaptive Histogram Equalization)
#      to enhance low-contrast lung detail
#   4. ImageNet normalisation (mean & std)
#   5. Light augmentation during training only:
#      ±10° rotation, horizontal flip, small translations

# ImageNet channel statistics (used because DenseNet-121 is pretrained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# CLAHE parameters — clip limit and tile grid size
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def apply_clahe(img_array: np.ndarray) -> np.ndarray:
    """Apply CLAHE to a grayscale numpy image and return 3-channel array."""
    # Ensure uint8
    if img_array.dtype != np.uint8:
        img_array = (img_array * 255).astype(np.uint8)
    enhanced = CLAHE.apply(img_array)
    # Replicate single channel to 3 channels (RGB expected by DenseNet-121)
    return np.stack([enhanced, enhanced, enhanced], axis=-1)


# torchvision transforms for TRAINING (with augmentation)
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(degrees=10),           # ±10° rotation
    transforms.RandomHorizontalFlip(),               # horizontal flip
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # small translation
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# torchvision transforms for VALIDATION / TEST (no augmentation)
val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

#CELL 7 — Dataset Class

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CELL 7 — Dataset Class  (FIXED)                    ║
# ╚══════════════════════════════════════════════════════════════╝

# BUG: Image path used subject_str[:2] giving "10" instead of "p10"
# Actual MIMIC-CXR-JPG structure:
#   files/p10/p10000032/s50414267/02aa804e-....jpg
#         ↑ must be "p10" not "10"

class MIMICCXRDataset(Dataset):
    def __init__(self, dataframe, tokenizer, report_cache,
                 image_dir, transform=None):
        self.df           = dataframe.reset_index(drop=True)
        self.tokenizer    = tokenizer
        self.report_cache = report_cache
        self.image_dir    = image_dir
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def _get_image_path(self, row):
        """
        Correct MIMIC-CXR-JPG path:
          files/p{prefix}/p{subject_id}/s{study_id}/{dicom_id}.jpg
          e.g. files/p10/p10000032/s50414267/02aa804e-bde0afdd-....jpg
        """
        subject_str = str(row['subject_id'])
        p_prefix    = 'p' + subject_str[:2]          # FIXED: 'p10' not '10'
        return os.path.join(
            self.image_dir,
            p_prefix,                                 # p10
            f'p{subject_str}',                        # p10000032
            f's{row["study_id"]}',                    # s50414267
            f'{row["dicom_id"]}.jpg'                  # 02aa804e-....jpg
        )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load and preprocess image
        img_path = self._get_image_path(row)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
        img = apply_clahe(img)
        if self.transform:
            img = self.transform(img)

        # Tokenize report
        key  = (row['subject_id'], row['study_id'])
        text = self.report_cache.get(key, "no findings")  # FIXED: better fallback
        encoding = self.tokenizer(
            text,
            max_length=MAX_TEXT_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids      = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        # Labels
        labels = torch.tensor(row[CLASSES].values.astype(np.float32))
        return img, input_ids, attention_mask, labels

#CELL 8 — Tokenizer & DataLoaders

In [ ]:
# Load ClinicalBERT tokenizer from Hugging Face Model Hub.
# "emilyalsentzer/Bio_ClinicalBERT" is the publicly released checkpoint
# described by Alsentzer et al. (2019) [Reference 7 in the paper].

CLINICALBERT_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(CLINICALBERT_NAME)
print(f"Tokenizer loaded: {CLINICALBERT_NAME}")

# Split DataFrame into train / val / test subsets
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'validate']
test_df  = df[df['split'] == 'test']

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Create Dataset objects
train_dataset = MIMICCXRDataset(train_df, tokenizer, report_cache, IMAGE_DIR, train_transform)
val_dataset   = MIMICCXRDataset(val_df,   tokenizer, report_cache, IMAGE_DIR, val_transform)
test_dataset  = MIMICCXRDataset(test_df,  tokenizer, report_cache, IMAGE_DIR, val_transform)

# Create DataLoaders (num_workers=2 works well in Colab)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("DataLoaders ready.")

#CELL 9 — Model Architecture (Late Fusion)

In [ ]:
# Architecture (from Section 3.3 of the paper):
#
#  Image Branch  → DenseNet-121 (ImageNet pretrained, GAP head removed)
#                  Output: 1,024-dim feature vector
#
#  Text Branch   → ClinicalBERT ([CLS] token representation)
#                  Output: 768-dim contextual embedding
#
#  Fusion Head   → Concatenation → 1,792-dim joint vector
#                  → FC(1792, 512) + ReLU
#                  → Dropout(0.3)
#                  → FC(512, 5)  + Sigmoid  (one sigmoid per disease)
#
#  Loss          → Weighted Binary Cross-Entropy (handles class imbalance)

class MultimodalFusionModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=DROPOUT_RATE):
        super().__init__()

        # ── IMAGE BRANCH: DenseNet-121 ─────────────────────────
        # Load ImageNet pretrained DenseNet-121
        densenet = models.densenet121(pretrained=True)
        # Remove the original classifier (1000-class FC layer)
        # and keep only the feature extraction layers
        self.image_encoder = densenet.features       # outputs [B, 1024, 7, 7]
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)  # → [B, 1024, 1, 1]
        image_feat_dim = 1024

        # ── TEXT BRANCH: ClinicalBERT ──────────────────────────
        # Load the pretrained ClinicalBERT transformer
        self.text_encoder = AutoModel.from_pretrained(CLINICALBERT_NAME)
        text_feat_dim = 768   # hidden size of BERT-base

        # ── FUSION HEAD ────────────────────────────────────────
        joint_dim = image_feat_dim + text_feat_dim   # 1,024 + 768 = 1,792
        self.fusion_head = nn.Sequential(
            nn.Linear(joint_dim, FC_HIDDEN),    # FC: 1792 → 512
            nn.ReLU(),
            nn.Dropout(dropout),                # regularisation
            nn.Linear(FC_HIDDEN, num_classes),  # FC: 512 → 5
            nn.Sigmoid()                        # per-class probability in [0,1]
        )

    def forward(self, images, input_ids, attention_mask):
        # ── Image forward pass ────────────────────────────────
        img_feats = self.image_encoder(images)           # [B, 1024, 7, 7]
        img_feats = self.global_avg_pool(img_feats)      # [B, 1024, 1, 1]
        img_feats = img_feats.view(img_feats.size(0), -1)  # [B, 1024]

        # ── Text forward pass ─────────────────────────────────
        text_output = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # Use [CLS] token (first token) as the sentence representation
        cls_embedding = text_output.last_hidden_state[:, 0, :]  # [B, 768]

        # ── Late fusion: concatenate then classify ─────────────
        joint = torch.cat([img_feats, cls_embedding], dim=1)  # [B, 1792]
        logits = self.fusion_head(joint)                       # [B, 5]
        return logits


# Instantiate model and move to GPU/CPU
model = MultimodalFusionModel().to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

#CELL 10 — Loss, Optimiser & Class Weights

In [ ]:
# The paper uses Weighted Binary Cross-Entropy to address the pronounced
# class imbalance in MIMIC-CXR (some diseases are rare).
# pos_weight tells BCEWithLogitsLoss how much to penalise false negatives:
#   pos_weight[c] = (# negative samples for class c) / (# positive samples for class c)

# Compute positive-class weights from the training split
pos_counts = train_df[CLASSES].sum()
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor((neg_counts / pos_counts).values, dtype=torch.float32).to(DEVICE)
print("Positive class weights:", pos_weight)

# BCEWithLogitsLoss combines Sigmoid + BCE in a numerically stable way.
# NOTE: Since the model's forward() already applies Sigmoid, we use plain BCELoss here.
criterion = nn.BCELoss()

# Adam optimiser at learning rate 1e-4 (as specified in the paper)
optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler — reduce LR by 0.5 if val loss plateaus for 3 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode='min', patience=3, factor=0.5, verbose=True
)

#CELL 11 — Training & Validation Loop

In [ ]:
# Trains the model for up to NUM_EPOCHS epochs.
# Early stopping is triggered if val AUROC does not improve for 5 epochs.
# Best model weights are saved to disk to avoid losing progress.

BEST_MODEL_PATH = 'best_multimodal_model.pth'
EARLY_STOP_PATIENCE = 5   # number of epochs to wait before stopping

def compute_auroc(labels_all, probs_all):
    """Compute mean AUROC across all 5 classes (macro average)."""
    aurocs = []
    for i in range(NUM_CLASSES):
        try:
            auroc = roc_auc_score(labels_all[:, i], probs_all[:, i])
            aurocs.append(auroc)
        except ValueError:
            # Happens if a class has no positive samples in the batch
            pass
    return np.mean(aurocs) if aurocs else 0.0


def run_epoch(loader, mode='train'):
    """
    Run one epoch in either 'train' or 'eval' mode.
    Returns average loss and mean AUROC for the epoch.
    """
    is_train = (mode == 'train')
    model.train() if is_train else model.eval()

    epoch_loss = 0.0
    all_labels = []
    all_probs  = []

    with torch.set_grad_enabled(is_train):
        for images, input_ids, attention_mask, labels in tqdm(loader, desc=mode):
            images         = images.to(DEVICE)
            input_ids      = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            labels         = labels.to(DEVICE)

            if is_train:
                optimiser.zero_grad()

            probs = model(images, input_ids, attention_mask)   # [B, 5]
            loss  = criterion(probs, labels)

            if is_train:
                loss.backward()
                optimiser.step()

            epoch_loss += loss.item() * images.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_probs.append(probs.detach().cpu().numpy())

    all_labels = np.vstack(all_labels)
    all_probs  = np.vstack(all_probs)
    avg_loss   = epoch_loss / len(loader.dataset)
    mean_auroc = compute_auroc(all_labels, all_probs)
    return avg_loss, mean_auroc


# ── Training loop ─────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_auroc': [], 'val_auroc': []}
best_val_auroc = 0.0
no_improve_count = 0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{NUM_EPOCHS}")

    train_loss, train_auroc = run_epoch(train_loader, mode='train')
    val_loss,   val_auroc   = run_epoch(val_loader,   mode='eval')

    scheduler.step(val_loss)  # adjust LR based on validation loss

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auroc'].append(train_auroc)
    history['val_auroc'].append(val_auroc)

    print(f"  Train → Loss: {train_loss:.4f} | AUROC: {train_auroc:.4f}")
    print(f"  Val   → Loss: {val_loss:.4f}   | AUROC: {val_auroc:.4f}")

    # Save model if validation AUROC improved
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        no_improve_count = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  ✔ New best model saved (AUROC = {best_val_auroc:.4f})")
    else:
        no_improve_count += 1
        print(f"  No improvement for {no_improve_count} epoch(s).")

    # Early stopping check
    if no_improve_count >= EARLY_STOP_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print(f"\nTraining complete. Best Val AUROC: {best_val_auroc:.4f}")

#CELL 12 — Plot Training Curves

In [ ]:
# Reproduce Figure 5 from the paper:
#   (a) Training & Validation Loss over epochs
#   (b) Training & Validation AUROC over epochs

epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Loss curves
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='orange')
axes[0].set_title('(a) Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend()
axes[0].grid(True)

# (b) AUROC curves
axes[1].plot(epochs_range, history['train_auroc'], label='Train AUROC', color='steelblue')
axes[1].plot(epochs_range, history['val_auroc'],   label='Val AUROC',   color='orange')
axes[1].set_title('(b) Training & Validation AUROC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean AUROC')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Figure 5 — Training Curves (Proposed Late-Fusion Model)', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print("Training curves saved to training_curves.png")

#CELL 13 — Test Set Evaluation

In [ ]:
# Load the best saved model weights and evaluate on the held-out test set.
# Reports per-class AUROC, Precision, Recall, and F1-Score
# (matching Table 2 in the paper).

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()
print("Best model loaded for test evaluation.")

all_labels = []
all_probs  = []

with torch.no_grad():
    for images, input_ids, attention_mask, labels in tqdm(test_loader, desc='Testing'):
        images         = images.to(DEVICE)
        input_ids      = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        probs = model(images, input_ids, attention_mask)
        all_labels.append(labels.numpy())
        all_probs.append(probs.detach().cpu().numpy())

all_labels = np.vstack(all_labels)   # [N, 5]
all_probs  = np.vstack(all_probs)    # [N, 5]

# Convert probabilities to binary predictions using threshold = 0.5
all_preds = (all_probs >= 0.5).astype(int)

# ── Per-class metrics ─────────────────────────────────────────
print("\n" + "="*60)
print("Per-Class Test Results (Table 2 in the paper)")
print("="*60)
results = []
for i, cls in enumerate(CLASSES):
    auroc     = roc_auc_score(all_labels[:, i], all_probs[:, i])
    f1        = f1_score(all_labels[:, i], all_preds[:, i], zero_division=0)
    report    = classification_report(
        all_labels[:, i], all_preds[:, i],
        target_names=['Neg', 'Pos'], output_dict=True, zero_division=0
    )
    precision = report['Pos']['precision']
    recall    = report['Pos']['recall']
    results.append({'Disease': cls, 'AUROC': auroc, 'Precision': precision,
                    'Recall': recall, 'F1-Score': f1})
    print(f"{cls:<20} AUROC={auroc:.4f}  P={precision:.4f}  R={recall:.4f}  F1={f1:.4f}")

# Mean across all classes
mean_row = {
    'Disease'  : 'Mean (5 classes)',
    'AUROC'    : np.mean([r['AUROC']     for r in results]),
    'Precision': np.mean([r['Precision'] for r in results]),
    'Recall'   : np.mean([r['Recall']    for r in results]),
    'F1-Score' : np.mean([r['F1-Score']  for r in results]),
}
results.append(mean_row)
print("-"*60)
print(f"{'Mean (5 classes)':<20} AUROC={mean_row['AUROC']:.4f}  "
      f"P={mean_row['Precision']:.4f}  R={mean_row['Recall']:.4f}  "
      f"F1={mean_row['F1-Score']:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv('test_results.csv', index=False)
print("\nResults saved to test_results.csv")

#CELL 14 — Per-Class AUROC Bar Chart

In [ ]:
# Reproduce Figure 3 from the paper:
# Bar chart comparing per-class AUROC of the proposed model vs baselines.
# NOTE: baseline values below are from Table 3 in the paper.
# You can replace them with your own baseline run results.

# Expected final AUROC per class from the paper (proposed model)
proposed_aurocs = [r['AUROC'] for r in results[:-1]]

# Approximate per-class values for baselines (read from paper Figure 3)
# Replace these with actual values from your baseline runs
image_only_aurocs = [0.87, 0.89, 0.86, 0.85, 0.77]  # DenseNet-121 only (approx)
text_only_aurocs  = [0.88, 0.90, 0.87, 0.85, 0.78]  # ClinicalBERT only (approx)

x       = np.arange(NUM_CLASSES)
width   = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, image_only_aurocs, width, label='Image-only (DenseNet-121)', color='skyblue')
ax.bar(x,          text_only_aurocs,  width, label='Text-only (ClinicalBERT)',  color='orange')
ax.bar(x + width,  proposed_aurocs,   width, label='Late Fusion (Proposed)',    color='green')

ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=20, ha='right')
ax.set_ylim(0.6, 1.0)
ax.set_ylabel('AUROC')
ax.set_title('Figure 3 — Per-Class AUROC: Proposed vs Baselines')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('per_class_auroc.png', dpi=150)
plt.show()
print("Per-class AUROC chart saved to per_class_auroc.png")

#CELL 15 — Confusion Matrix

In [ ]:
# Reproduce Figure 6 from the paper.
# Since this is a multi-label problem, we plot a binary confusion matrix
# for each class separately, then display them in a grid.

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, cls in enumerate(CLASSES):
    cm = confusion_matrix(all_labels[:, i], all_preds[:, i])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
        xticklabels=['Pred Neg', 'Pred Pos'],
        yticklabels=['True Neg', 'True Pos']
    )
    axes[i].set_title(f'{cls}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

# Hide the unused 6th subplot
axes[-1].set_visible(False)

plt.suptitle('Figure 6 — Per-Class Confusion Matrices (Test Set)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()
print("Confusion matrices saved to confusion_matrices.png")

#CELL 16 — Model Comparison Table (Table 3)

In [ ]:
# Reproduce Table 3 and Figure 4 from the paper.
# These baseline values are taken directly from Table 3 in the paper.
# To reproduce fully, you should separately train each baseline model
# using the same train/val/test split and report their metrics here.

comparison_data = {
    'Model'   : ['ResNet-50', 'DenseNet-121 (CheXNet)', 'ClinicalBERT',
                 'Early Fusion (raw concat)', 'Late Fusion (Proposed)'],
    'Modality': ['Image', 'Image', 'Text', 'Image + Text', 'Image + Text'],
    'AUROC'   : [0.81, 0.85, 0.86, 0.87, mean_row['AUROC']],
    'F1'      : [0.70, 0.74, 0.76, 0.79, mean_row['F1-Score']],
    'Params'  : ['25.6 M', '8.0 M', '110 M', '118 M', '118 M'],
}
comparison_df = pd.DataFrame(comparison_data)
print("\nTable 3 — Model Comparison")
print(comparison_df.to_string(index=False))
comparison_df.to_csv('model_comparison.csv', index=False)

# Bar chart of AUROC and F1 for all five models (Figure 4)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']
models_short = ['ResNet-50', 'DenseNet-121', 'ClinicalBERT', 'Early Fusion', 'Late Fusion']

axes[0].bar(models_short, comparison_data['AUROC'], color=colors)
axes[0].set_ylim(0.7, 1.0)
axes[0].set_ylabel('Mean AUROC')
axes[0].set_title('Mean AUROC Across Models')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(comparison_data['AUROC']):
    axes[0].text(i, v + 0.003, f'{v:.2f}', ha='center', fontsize=10)

axes[1].bar(models_short, comparison_data['F1'], color=colors)
axes[1].set_ylim(0.6, 0.95)
axes[1].set_ylabel('Mean F1-Score')
axes[1].set_title('Mean F1-Score Across Models')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(comparison_data['F1']):
    axes[1].text(i, v + 0.003, f'{v:.2f}', ha='center', fontsize=10)

plt.suptitle('Figure 4 — Model Comparison (Table 3)', fontsize=14)
plt.tight_layout()
plt.savefig('model_comparison_chart.png', dpi=150)
plt.show()
print("Model comparison chart saved to model_comparison_chart.png")

#CELL 17 — Class Distribution Plot (Figure 1)

In [ ]:
# Reproduce Figure 1 from the paper:
# Bar chart showing the number of positive samples per class
# in the training set, illustrating the class imbalance.

class_counts = train_df[CLASSES].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
bars = plt.bar(class_counts.index, class_counts.values,
               color=['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0'])
plt.title('Figure 1 — Class Distribution in MIMIC-CXR Training Set')
plt.ylabel('Number of Positive Samples')
plt.xlabel('Thoracic Disease')
plt.xticks(rotation=15)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 500,
             f'{int(bar.get_height()):,}',
             ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()
print("Class distribution chart saved to class_distribution.png")

#CELL 18 — Save Final Model & Summary

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CELL 18 — Save Final Model & Summary  (FIXED)      ║
# ╚══════════════════════════════════════════════════════════════╝

# BUG: Original code saved model only to Colab local disk
# Colab resets every session — model would be permanently lost
# FIX: Always copy final weights to Google Drive

FINAL_MODEL_PATH = 'final_multimodal_model.pth'
torch.save(model.state_dict(), FINAL_MODEL_PATH)
print(f"Model saved locally: {FINAL_MODEL_PATH}")

# Mount Drive and copy model there so it survives session resets
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil
drive_save_path = '/content/drive/MyDrive/multimodal_cxr_model.pth'
shutil.copy(FINAL_MODEL_PATH, drive_save_path)
print(f"✔ Model safely saved to Google Drive: {drive_save_path}")

# Also save results CSVs to Drive
shutil.copy('test_results.csv',      '/content/drive/MyDrive/test_results.csv')
shutil.copy('model_comparison.csv',  '/content/drive/MyDrive/model_comparison.csv')

print("\n" + "="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
print(f"Model      : Late-Fusion (DenseNet-121 + ClinicalBERT)")
print(f"Dataset    : MIMIC-CXR v2.0.0 — 5 disease classes")
print(f"Test AUROC : {mean_row['AUROC']:.4f}")
print(f"Test F1    : {mean_row['F1-Score']:.4f}")
print(f"Test Prec  : {mean_row['Precision']:.4f}")
print(f"Test Recall: {mean_row['Recall']:.4f}")
print("\nOutput files generated:")
for f in ['training_curves.png', 'per_class_auroc.png',
          'confusion_matrices.png', 'model_comparison_chart.png',
          'class_distribution.png', 'test_results.csv',
          'model_comparison.csv', BEST_MODEL_PATH, FINAL_MODEL_PATH]:
    print(f"  ✔ {f}")
print("="*60)